# Drone Human Detection — Notebook 2 of 3
## Task 02: Model Training
**Antlings AI/ML Internship — Technical Assessment**

This notebook trains YOLOv8n on the filtered VisDrone dataset
prepared in Notebook 1. It requires a T4 GPU runtime.
Go to Runtime → Change Runtime Type → T4 GPU before running.

In [ ]:
!pip install ultralytics -q

import torch
from ultralytics import YOLO

print(f"Ultralytics ready ✅")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics ready ✅
CUDA available: True
GPU: Tesla T4


## Cell 2 — Mount Drive & Restore Dataset
Mount Google Drive and unzip the filtered dataset saved
in Notebook 1 to Colab's local SSD. Local SSD is used
for faster I/O during training — reading 6,000+ images
directly from Drive during each epoch would be too slow.

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/Antlings_Drone_Project'

# Unzip dataset back into Colab's fast local storage
print("⏳ Unzipping dataset to local storage...")
!unzip -q "{DRIVE_PROJECT}/visdrone_yolo.zip" -d /
print("✅ Dataset restored at /content/visdrone_yolo/")

# Confirm structure (correct paths)
print("\n--- Dataset Structure ---")
for split in ['train', 'val']:
    imgs = len(os.listdir(f'/content/visdrone_yolo/images/{split}'))
    labs = len(os.listdir(f'/content/visdrone_yolo/labels/{split}'))
    print(f"{split}: {imgs} images | {labs} label files")

Mounted at /content/drive
⏳ Unzipping dataset to local storage...
✅ Dataset restored at /content/visdrone_yolo/

--- Dataset Structure ---
train: 6468 images | 6468 label files
val: 548 images | 548 label files


## Cell 3 — Create data.yaml
YOLO requires a configuration file specifying dataset paths
and class names. We generate this programmatically to avoid
manual errors. The file defines 2 classes: human (0) and car (1),
matching the remapping done in Notebook 1.

In [ ]:
import os

yaml_content = """path: /content/visdrone_yolo
train: images/train
val: images/val

nc: 2
names:
  0: human
  1: car
"""

with open('/content/visdrone_yolo/data.yaml', 'w') as f:
    f.write(yaml_content)

print("✅ data.yaml created")
print(open('/content/visdrone_yolo/data.yaml').read())

✅ data.yaml created
path: /content/visdrone_yolo
train: images/train
val: images/val

nc: 2
names:
  0: human
  1: car



## Cell 4 — Train YOLOv8n
Fine-tune YOLOv8n (nano) on our filtered VisDrone dataset.

Key decisions:
- YOLOv8n: lightweight (5.9MB), fast to train, deployable on edge devices
- imgsz=416: better resolution for tiny aerial objects than default 640
- batch=32: fills T4 VRAM efficiently
- epochs=25: constrained by Colab T4 GPU quota
- Pretrained on COCO: transfer learning gives a strong starting point

The best checkpoint (highest val mAP) is saved automatically as best.pt.

In [ ]:

from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/visdrone_yolo/data.yaml',
    epochs=25,
    imgsz=416,
    batch=32,
    workers=2,
    device=0,
    patience=5,
    save=True,
    project='/content/runs',
    name='visdrone_yolov8n',
    exist_ok=True,
    verbose=True
)

print("\n✅ Training complete!")
print(f"Best weights: /content/runs/visdrone_yolov8n/weights/best.pt")

Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/visdrone_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=visdrone_yolov8n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pat

## Cell 5 — Save Weights & Results to Drive
Copy best.pt and the full training run folder to Google Drive
for persistent storage. Colab's local filesystem resets when
the session ends — saving to Drive ensures weights and results
survive across sessions and are available for Notebook 3.

In [ ]:
import shutil
import os

DRIVE_PROJECT = '/content/drive/MyDrive/Antlings_Drone_Project'
RUN_DIR = '/content/runs/visdrone_yolov8n'  # changed s → n


shutil.copy(
    f'{RUN_DIR}/weights/best.pt',
    f'{DRIVE_PROJECT}/best.pt'
)
print("✅ best.pt saved to Drive")


print("⏳ Zipping evaluation results...")
shutil.make_archive(
    base_name=f'{DRIVE_PROJECT}/runs_visdrone_yolov8n',  # changed s → n
    format='zip',
    root_dir='/content/runs',
    base_dir='visdrone_yolov8n'  # changed s → n
)
print("✅ Evaluation results saved to Drive!")

size_mb = os.path.getsize(f"{DRIVE_PROJECT}/best.pt") / (1024 * 1024)
print(f"\n--- Verification ---")
print(f"best.pt size: {size_mb:.1f} MB")
print(f"\n🎯 Colab 2 complete. Safe to close.")

✅ best.pt saved to Drive
⏳ Zipping evaluation results...
✅ Evaluation results saved to Drive!

--- Verification ---
best.pt size: 5.9 MB

🎯 Colab 2 complete. Safe to close.
